In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import sys
sys.path.append("../src")

In [12]:
PDF_PATH = pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = model_name = "gemma2:2b"

# Ontology modes:
#   None       -> no ontology
#   "minimal"  -> minimal fallback ontology
#   path       -> real ontology file
SEED_ONTOLOGY_INPUT = str("../data/ontology/ContextOntology-COInd4.owl")
ONTOLOGY_PATH = "../data/ontology/ContextOntology-COInd4.owl"

In [ ]:
!python -m nltk.downloader wordnet

#### Layer 1

In [ ]:
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf
from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend
from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer

pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
model_name = "gemma2:27b-instruct-q4_K_M"

raw_text = extract_text_from_pdf(pdf_path)

document = Document(
    doc_id="pdf_test_001",
    source_path=pdf_path,
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level="Treat machine types, failure types, and symptom types as concepts; treat document-specific occurrences as individuals.",
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=model_name,
    user_guidance=guidance,
)

ollama = OllamaBackend()

translator = DeepTranslatorBackend(max_chars=4000)

pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=True,          # enable translation
            translator=translator,   # use free translator
            source_language="fr",    # for example
            target_language="en",
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=2,
            temperature=0.0,
        ),
    ]
)


runner = Runner(pipeline)
final_state = runner.run(state)

final_state.logs

In [ ]:
len(final_state.document.chunks)

In [ ]:
for expr in final_state.linguistic_expressions[:10]:
    print(expr)

#### Layer 3

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# NeoOLAF imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
PDF_PATH = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = "gemma2:27b-instruct-q4_K_M"

# Debug limits
MAX_CHUNKS = 2
MAX_EXPRESSIONS = 5

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

# ---------------------------------------------------------
# Optional semantic guidance
# ---------------------------------------------------------
guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

# ---------------------------------------------------------
# State and backends
# ---------------------------------------------------------
state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
        ),
    ]
)

runner = Runner(pipeline)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect one enriched expression
# ---------------------------------------------------------
item = final_state.enriched_expressions[0]

print("Base expression:", item.base_expression.text)
print("Aliases:", item.aliases)
print("Alias sources:")
pprint(item.alias_sources)

print("\nSynonyms:", item.synonyms)
print("Synonym sources:")
pprint(item.synonym_sources)

print("\nLexical variants:", item.lexical_variants)
print("Lexical variant sources:")
pprint(item.lexical_variant_sources)

print("\nDefinition:", item.definition)
print("Ontology hints:", item.ontology_hints)

print("\nEvidence sources:")
for ev in item.enrichment_evidence:
    print("-", ev.source, "|", ev.reference)

In [ ]:
final_state.enriched_expressions

#### Layer 4

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)
from neoolaf.layers.layer03_candidate_typing_resolution.component import (
    CandidateTypingResolutionLayer,
)
from neoolaf.layers.layer04_candidate_relation_extraction.component import (
    CandidateRelationExtractionLayer,
)

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = None#20

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
        ),
    ]
)

runner = Runner(pipeline)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 4 outputs
# ---------------------------------------------------------
print("Candidate relation assertions:", len(final_state.candidate_relation_assertions))

for item in final_state.candidate_relation_assertions[:10]:
    print("=" * 80)
    print("Assertion ID:", item.assertion_id)
    print("Relation:", item.relation_label, "|", item.relation_candidate_id)
    print("Source:", item.source_candidate_label, "|", item.source_candidate_id, "|", item.source_candidate_type)
    print("Target:", item.target_candidate_label, "|", item.target_candidate_id, "|", item.target_candidate_type)
    print("Chunk:", item.chunk_id)
    print("Confidence:", item.confidence)
    print("Justification:", item.justification)

#### Layer 5

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)
from neoolaf.layers.layer03_candidate_typing_resolution.component import (
    CandidateTypingResolutionLayer,
)
from neoolaf.layers.layer04_candidate_relation_extraction.component import (
    CandidateRelationExtractionLayer,
)
from neoolaf.layers.layer05_candidate_triple_generation.component import (
    CandidateTripleGenerationLayer,
)

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = None#20
MAX_ASSERTIONS = None

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=MAX_ASSERTIONS,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(
    pipeline=pipeline,
    runs_root="runs",
    verbose=True,
)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 5 outputs
# ---------------------------------------------------------
print("Candidate triples:", len(final_state.candidate_triples))

for triple in final_state.candidate_triples[:10]:
    print("=" * 90)
    print("Triple ID:", triple.triple_id)
    print("S:", triple.subject_label, "|", triple.subject_id, "|", triple.subject_type)
    print("P:", triple.predicate_label, "|", triple.predicate_id)
    print("O:", triple.object_label, "|", triple.object_id, "|", triple.object_type)
    print("Chunk:", triple.chunk_id)
    print("Confidence:", triple.confidence)
    print("Justification:", triple.justification)

In [ ]:
print(len(final_state.document.chunks))
print(len(final_state.linguistic_expressions))
print(len(final_state.enriched_expressions))
print(len(final_state.entity_candidates))
print(len(final_state.relation_candidates))
print(len(final_state.attribute_candidates))
print(len(final_state.event_candidates))

#### Layer 6

In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 6 outputs
# ---------------------------------------------------------
print("Concept candidates:", len(final_state.concept_candidates))
print("Ontology relation candidates:", len(final_state.ontology_relation_candidates))

if final_state.concept_candidates:
    print("\nFirst concept candidate:")
    pprint(final_state.concept_candidates[0])

if final_state.ontology_relation_candidates:
    print("\nFirst ontology relation candidate:")
    pprint(final_state.ontology_relation_candidates[0])

#### Layer 7

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 7 outputs
# ---------------------------------------------------------
print("Concept hierarchy links:", len(final_state.concept_hierarchy_links))
print("Relation hierarchy links:", len(final_state.relation_hierarchy_links))

if final_state.concept_hierarchy_links:
    print("\nFirst concept hierarchy link:")
    pprint(final_state.concept_hierarchy_links[0])

if final_state.relation_hierarchy_links:
    print("\nFirst relation hierarchy link:")
    pprint(final_state.relation_hierarchy_links[0])

#### Layer 8

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 8 outputs
# ---------------------------------------------------------
print("Axiom schema candidates:", len(final_state.axiom_schema_candidates))

if final_state.axiom_schema_candidates:
    print("\nFirst axiom schema candidate:")
    pprint(final_state.axiom_schema_candidates[0])

#### Layer 9

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=ollama,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 9 outputs
# ---------------------------------------------------------
print("General axiom candidates:", len(final_state.general_axiom_candidates))

if final_state.general_axiom_candidates:
    print("\nFirst general axiom candidate:")
    pprint(final_state.general_axiom_candidates[0])

#### Layer 10

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer
from neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=ollama,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        ValidationReasoningLayer(
            max_triples=None,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 10 outputs
# ---------------------------------------------------------
print("Validation report:")
pprint(final_state.validation_report)

print("\nReasoning report notes:")
pprint(final_state.reasoning_report.notes if final_state.reasoning_report else [])

print("\nNumber of inferred triples:")
print(len(final_state.reasoning_report.inferred_triples) if final_state.reasoning_report else 0)

#### Layer 11 + 12

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.ontology.factory import build_seed_ontology

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer
from neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer
from neoolaf.layers.layer11_inference_completion.component import InferenceCompletionLayer
from neoolaf.layers.layer12_serialization.component import SerializationLayer
from neoolaf.domain.user_guidance import (
    UserGuidance,
    TypingExample,
    RelationExample,
    PromotionExample,
    NegativeExample,
)


# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 1 #2
MAX_EXPRESSIONS = 2 #20
MAX_RELATION_MENTIONS = 2 #20

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

# ---------------------------------------------------------
# User guidance
# ---------------------------------------------------------
guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",

    # New: ontology depth
    ontology_depth="deep",

    # New: examples
    typing_examples=[
        TypingExample(
            text="bearing",
            expected_type="entity",
            explanation="A machine component should be treated as an entity."
        ),
        TypingExample(
            text="overheating",
            expected_type="event",
            explanation="A degradation/failure process should be treated as an event/state."
        ),
        TypingExample(
            text="emitted by",
            expected_type="relation",
            explanation="A linking phrase between an alarm and a device is a relation."
        ),
    ],
    relation_examples=[
        RelationExample(
            text="The alarm is emitted by the PLC.",
            source_label="alarm",
            relation_label="emittedBy",
            target_label="PLC",
            explanation="This is a direct emitted-by relation."
        ),
    ],
    promotion_examples=[
        PromotionExample(
            text="overheating",
            promote=True,
            promoted_label="OverheatingEvent",
            explanation="A recurring failure/state pattern should become a concept."
        ),
        PromotionExample(
            text="emitted by",
            promote=True,
            promoted_label="emittedBy",
            explanation="A reusable linking phrase should become an ontology relation."
        ),
    ],
    negative_examples=[
        NegativeExample(
            text="section 8.1",
            explanation="Section numbering is not a semantic concept.",
            target_layer="layer01",
        ),
        NegativeExample(
            text="page 12",
            explanation="Page references should not be promoted to ontology concepts.",
            target_layer="layer06",
        ),
    ],

    # New: executable policy
    promotion_min_confidence=0.45,
    hierarchy_min_confidence=0.40,
    concept_promotion_bias=0.70,
)

# ---------------------------------------------------------
# Seed ontology
# ---------------------------------------------------------
seed_ontology = build_seed_ontology(SEED_ONTOLOGY_INPUT)

print("Seed ontology loaded:", seed_ontology is not None)
if seed_ontology is not None:
    print("Ontology URI:", seed_ontology.ontology_uri)
    print("Ontology label:", seed_ontology.ontology_label)
    print("Classes:", len(seed_ontology.get_classes()))
    print("Properties:", len(seed_ontology.get_properties()))

# ---------------------------------------------------------
# Pipeline state
# ---------------------------------------------------------
state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
    seed_ontology=seed_ontology,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=True,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=ollama,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        ValidationReasoningLayer(
            max_triples=None,
            verbose=True,
        ),
        InferenceCompletionLayer(
            max_inferred_triples=None,
            verbose=True,
        ),
        SerializationLayer(
            output_subdir="data/exports",
            base_uri="http://neoolaf.org/resource/",
            verbose=True,
        ),
    ],
    verbose=True,
)

# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect outputs
# ---------------------------------------------------------
print("Completion candidates:", len(final_state.completion_candidates))
if final_state.completion_candidates:
    print("\nFirst completion candidate:")
    pprint(final_state.completion_candidates[0])

print("\nExport folder:")
export_dir = Path(final_state.artifact_dir) / "data" / "exports"
print(export_dir)

if export_dir.exists():
    print("\nExported files:")
    for path in sorted(export_dir.iterdir()):
        print("-", path.name)

#### FINAL VERSION

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner
from neoolaf.core.execution_config import ExecutionConfig

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import (
    UserGuidance,
    TypingExample,
    RelationExample,
    PromotionExample,
    NegativeExample,
)

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
#from neoolaf.resources.llm_backends.vllm_backend import VLLMBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.ontology.factory import build_seed_ontology

from neoolaf.grounding.rag.registry import RetrievalRegistry
from neoolaf.grounding.rag.engine import SemanticRAGEngine
from neoolaf.grounding.rag.adapters.neoolaf_semantic_rag_adapter import NeoOLAFSemanticRAGAdapter
from neoolaf.grounding.rag.spaces.ontology_space import OntologySpace
from neoolaf.grounding.rag.spaces.artifact_space import ArtifactSpace
from neoolaf.grounding.rag.spaces.web_space import WebSpace
from neoolaf.grounding.rag.spaces.wikidata_space import WikidataSpace
from neoolaf.grounding.rag.spaces.wikipedia_space import WikipediaSpace
from neoolaf.grounding.rag.spaces.wordnet_space import WordNetSpace

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer
from neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer
from neoolaf.layers.layer11_inference_completion.component import InferenceCompletionLayer
from neoolaf.layers.layer12_serialization.component import SerializationLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

# Use the full document
MAX_CHUNKS = None
MAX_EXPRESSIONS = None
MAX_RELATION_MENTIONS = None

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

# ---------------------------------------------------------
# User guidance
# ---------------------------------------------------------
guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
    ontology_depth="deep",
    typing_examples=[
        TypingExample(text="bearing", expected_type="entity"),
        TypingExample(text="overheating", expected_type="event"),
        TypingExample(text="emitted by", expected_type="relation"),
    ],
    relation_examples=[
        RelationExample(
            text="The alarm is emitted by the PLC.",
            source_label="alarm",
            relation_label="emittedBy",
            target_label="PLC",
        ),
    ],
    promotion_examples=[
        PromotionExample(text="overheating", promote=True, promoted_label="OverheatingEvent"),
        PromotionExample(text="emitted by", promote=True, promoted_label="emittedBy"),
    ],
    negative_examples=[
        NegativeExample(text="section 8.1", explanation="Not a semantic concept.", target_layer="layer01"),
    ],
)

# ---------------------------------------------------------
# Seed ontology
# ---------------------------------------------------------
seed_ontology = build_seed_ontology(str(ONTOLOGY_PATH))

# ---------------------------------------------------------
# Pipeline state
# ---------------------------------------------------------
state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
    seed_ontology=seed_ontology,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
#vllm = VLLMBackend(
#    host="http://localhost:8000",
#    api_key="dummy",
#    timeout=900,
#    max_retries=3,
#    retry_wait_seconds=5.0,
#)
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Build SemanticRAG registry and adapter
# ---------------------------------------------------------
registry = RetrievalRegistry()
registry.register(OntologySpace(state.seed_ontology))
registry.register(ArtifactSpace(state))
registry.register(WebSpace(web_search_source))
registry.register(WikidataSpace(wikidata_source))
registry.register(WikipediaSpace(wikipedia_source))
registry.register(WordNetSpace(wordnet_source))

semantic_rag_engine = SemanticRAGEngine(
    registry=registry,
    ollama_backend=ollama,
    model_name=MODEL_NAME,
)

rag_adapter = NeoOLAFSemanticRAGAdapter(semantic_rag_engine)

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=True,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=None,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=None,
            use_web_search=True,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=ollama,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        ValidationReasoningLayer(
            max_triples=None,
            verbose=True,
        ),
        InferenceCompletionLayer(
            max_inferred_triples=None,
            verbose=True,
        ),
        SerializationLayer(
            output_subdir="data/exports",
            base_uri="http://neoolaf.org/resource/",
            verbose=True,
        ),
    ],
    verbose=True,
)

# ---------------------------------------------------------
# Execution config for full-document iterative mode
# ---------------------------------------------------------
execution_config = ExecutionConfig(
    mode="chunk_iterative_mode",
    chunk_loop_enabled=True,
    chunk_layer_names=[
        "layer01_linguistic_expression_extraction",
        "layer02_candidate_enrichment",
        "layer03_candidate_typing_resolution",
        "layer04_candidate_relation_extraction",
        "layer05_candidate_triple_generation",
    ],
    global_layer_names=[
        "layer06_concept_relation_induction",
        "layer07_hierarchisation",
        "layer08_axiom_schemata_extraction",
        "layer09_general_axiom_extraction",
        "layer10_validation_reasoning",
        "layer11_inference_completion",
        "layer12_serialization",
    ],
    max_chunks=None,
)

# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
runner = Runner(
    pipeline=pipeline,
    runs_root="runs",
    verbose=True,
    execution_config=execution_config,
)

final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect
# ---------------------------------------------------------
print("Chunks in document:", len(final_state.document.chunks))
print("Merged linguistic expressions:", len(final_state.linguistic_expressions))
print("Merged enriched expressions:", len(final_state.enriched_expressions))
print("Merged candidate triples:", len(final_state.candidate_triples))
print("Concept candidates:", len(final_state.concept_candidates))
print("Axiom schemata:", len(final_state.axiom_schema_candidates))
print("Completion candidates:", len(final_state.completion_candidates))

print("\nExport folder:")
export_dir = Path(final_state.artifact_dir) / "data" / "exports"
print(export_dir)

if export_dir.exists():
    print("\nExported files:")
    for path in sorted(export_dir.iterdir()):
        print("-", path.name)

In [ ]:
final_state.reasoning_report.inferred_triples